In [ ]:
import pandas as pd
# import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

In [2]:
df = pd.read_csv('data/users_behavior.csv')

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB


In [4]:
df.head()

,calls,minutes,messages,mb_used,is_ultra
0,40.0,311.90,83.0,19915.42,0
1,85.0,516.75,56.0,22696.96,0
2,77.0,467.66,86.0,21060.45,0
3,106.0,745.53,81.0,8437.39,1
4,66.0,418.74,1.0,14502.75,0


In [5]:
# split df into train, validation and test
df_train, df_val_test = train_test_split(df, test_size=0.4, random_state=123)
df_val, df_test = train_test_split(df_val_test, test_size=0.5, random_state=123)

features_train = df_train.drop('is_ultra', axis=1)
target_train = df_train['is_ultra']

features_val = df_val.drop('is_ultra', axis=1)
target_val = df_val['is_ultra']

features_test = df_test.drop('is_ultra', axis=1)
target_test = df_test['is_ultra']

### Validation

#### DecisionTreeClassifier

In [6]:
score_train_01 = 0
score_val_01 = 0
best_depth = 0

# # Initiate a dictionary of lists to plot later
# model_01_scores = {
#     'depth': [],
#     'score_train': [],
#     'score_val': []
# }

for depth in range (1, 21):
    model_01 = DecisionTreeClassifier(max_depth=depth, random_state=123)
    model_01.fit(features_train, target_train)

    score_train = model_01.score(features_train, target_train)

    score_val = model_01.score(features_val, target_val)

    # model_01_scores['depth'].append(depth)
    # model_01_scores['score_train'].append(score_train)
    # model_01_scores['score_val'].append(score_val)

    if score_val > score_val_01:
        score_train_01 = score_train
        score_val_01 = score_val
        best_depth = depth

# model_01_df = pd.DataFrame(model_01_scores)

# best model
model_01 = DecisionTreeClassifier(max_depth=best_depth, random_state=123)
model_01.fit(features_train, target_train)

print(
    f'best_depth: {best_depth}, '
    f'train score:{score_train_01:.3f}, '
    f'val score: {score_val_01:.3f}'
)

best_depth: 5, train score:0.821, val score: 0.823


The best max depth obtained between 1 and 20 is 5, with 0.823 accuracy score with the validation data. 

The train score is close, 0.821, showing that there is no overfitting.

In [7]:
# fig = px.line(
#     model_01_df,
#     x='depth',
#     y=['score_train', 'score_val'],
#     title='Accuracy Score vs Max Depth for DecisionTreeClassifier',
#     labels={'depth': 'max depth', 'value': 'Accuracy Score'}
# )

# fig.add_vline(
#     x=best_depth, 
#     line_dash='dash', 
#     annotation_text=f'best depth: {best_depth}'
# )

# fig.show()

#### RandomForestClassifier

In [8]:
score_train_02 = 0
score_val_02 = 0 
best_depth = 0
best_n_estimators = 0
for estimators in range(1, 51):
    for depth in range(1, 21):
        model_02 = RandomForestClassifier(n_estimators=estimators, 
                                          max_depth=depth,
                                          random_state=123)
        model_02.fit(features_train, target_train)

        score_train = model_02.score(features_train, target_train)

        score_val = model_02.score(features_val, target_val)

        if score_val > score_val_02:
            score_train_02 = score_train
            score_val_02 = score_val
            best_depth = depth
            best_n_estimators = estimators

model_02 = RandomForestClassifier(n_estimators=best_n_estimators, 
                                  max_depth=best_depth,
                                  random_state=123)
model_02.fit(features_train, target_train)

print(
    f'best_depth: {best_depth}, '
    f'best_n_estimators: {best_n_estimators}, '
    f'train score: {score_train_02:.3f}, '
    f'val score: {score_val_02:.3f}'
)

best_depth: 10, best_n_estimators: 26, train score: 0.890, val score: 0.837


The best validation accuracy score is 0.837 for 26 estimators and 10 max depth. 

The train score is close, 0.890, showing no overfitting.

#### LogisticRegression

In [9]:
model_03 = LogisticRegression(random_state=123, solver='liblinear')

model_03.fit(features_train, target_train)

score_train_03 = model_03.score(features_train, target_train)

score_val_03 = model_03.score(features_val, target_val)

print(f'train score:{score_train_03:.3f}, val score: {score_val_03:.3f}')

train score:0.718, val score: 0.720


The validation score for Logistic Regression is 0.720, the worst of all three models trained.

### Model Selection

Select the model with the highest validation accuracy score.

In [10]:
models_dict = {
    score_val_01: model_01,
    score_val_02: model_02,
    score_val_03: model_03
}
model = models_dict[max(models_dict.keys())]

print(f'The best model trained is: {model}')

The best model trained is: RandomForestClassifier(max_depth=10, n_estimators=26, random_state=123)


### Testing

In [11]:
score_test = model.score(features_test, target_test)
print(f'Best model accuracy score with the test data: {score_test:.3f}')

Best model accuracy score with the test data: 0.798


In [12]:
print('Frequency:')
print(target_test.value_counts(normalize=True))

dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(features_train, target_train)
print()
print(f'Dummy score: {dummy.score(features_test, target_test):.3f}')


Frequency:
is_ultra
0    0.693624
1    0.306376
Name: proportion, dtype: float64

Dummy score: 0.694


The most frequent value fraction is 0.694, which is different from the model accuracy score, showing that the model is not just guessing the most frequent all the time.

### Conclusion

The best model for this dataset is RandomForestClassifier(max_depth=10, n_estimators=26, random_state=123), with 0.798 accuracy score obtained in the test.

The accuracy score is higher than the majority class baseline, 0.694.
